# Phase 5-C1 — Long Development Snapshot Integration & Readiness

[Open in Colab](https://colab.research.google.com/github/hh4832/strategy-research-agent/blob/feature/phase5c1-first-scientific-research/notebooks/Phase5C1_Long_Development_Readiness.ipynb)

**Configure only the first code cell.** It mounts Drive and contains every human
choice and confirmation. Fresh runtime / Run All is supported. Later cells do not
prompt. PREPARE_ONLY is the default and never switches itself to execution.

**Current blocker:** Core bc059 supports only one SEC/TAF interval per run. The
fixed 2017–2024 scope crosses rates. EXECUTE_AUTHORIZED therefore fails closed
before any batch, even with an authorization; this is not an executable long
backtest delivery. Do not issue authorization until this separately reviewed Core
capability and the real data pins are resolved. No conversion is performed.


In [ ]:
# ALL MANUAL OPERATIONS / SETTINGS — edit only this cell.
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive
drive.mount('/content/drive')

AGENT_REPOSITORY = 'https://github.com/hh4832/strategy-research-agent.git'
CORE_REPOSITORY = 'https://github.com/hh4832/market-research-core.git'
AGENT_SHA = "e31b8d00c89199e4c0407442c722934df9992701"
CORE_SHA = 'bc0592984a09d806ebb9ef78ce377db78b404f86'
MODE = 'PREPARE_ONLY'  # alternative: EXECUTE_AUTHORIZED; current cost blocker still refuses it
SNAPSHOT_ID = 'TSM_DEV_1M_20161019_20241231_ca9f87df83f31bd24211'
RECOVERY_ARCHIVE = Path('/content/drive/MyDrive/00Quant_Research/data/development_recovery_archives/TSM_DEV_1M_20161019_20241231_ca9f87df83f31bd24211_20261110T142034_335006Z')
# The verified recovery record preserves the original conversion approval dict.
# Alternatively select the original standalone personal approval JSON and set False.
ORIGINAL_APPROVAL_EVIDENCE_PATH = RECOVERY_ARCHIVE / 'recovery_provenance.json'
APPROVAL_IS_IN_RECOVERY_RECORD = True
ORIGINAL_CONVERSION_APPROVAL_REF = 'https://github.com/hh4832/market-research-core/pull/4#issuecomment-6768321310'

# FIRST RUN: None. Observations stay OBSERVED_NOT_TRUSTED, never auto-approved.
# AFTER independent review, personally paste all six SHA256 values from the review.
REVIEWED_DATA_PINS = None
PIN_REVIEW_EVIDENCE_REF = ''

NEW_AUTHORIZATION_ID = 'RA-IBKR_TSM-SMA520_P5C1-0001'
EXECUTOR_ID = 'colab-hh4832-exec-01'
# A later formal authorization lives in separately reviewed governance books.
# None below uses this pinned Agent checkout's existing books (currently no C1 authorization).
GOVERNANCE_ROOT = None
PREVIOUS_PREPARATION_DIRECTORY = None  # required for EXECUTE_AUTHORIZED; never generate a new request implicitly
BIND_AUTHORIZATION_URL = ''  # research lead's own GitHub comment, not posted here
APPROVE_TASK_URL = ''        # research lead's own second GitHub comment

LOCAL_PARENT = Path('/content/phase5c1')
DRIVE_OUTPUT_PARENT = Path('/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1')
CONFIRM_SETTINGS = False  # personally confirm the paths, mode and evidence above

import re
assert MODE in ('PREPARE_ONLY','EXECUTE_AUTHORIZED')
assert CONFIRM_SETTINGS, 'STOP: review and confirm this first code cell; no source conversion or batch performed'
assert re.fullmatch('[0-9a-f]{40}',AGENT_SHA) and re.fullmatch('[0-9a-f]{40}',CORE_SHA)
assert RECOVERY_ARCHIVE.is_dir() and ORIGINAL_APPROVAL_EVIDENCE_PATH.is_file()
assert LOCAL_PARENT.resolve().is_relative_to(Path('/content').resolve())
assert not LOCAL_PARENT.resolve().is_relative_to(Path('/content/drive').resolve()), 'STOP: staging must use local disk'
assert DRIVE_OUTPUT_PARENT.resolve().is_relative_to(Path('/content/drive').resolve())
if REVIEWED_DATA_PINS is not None:
    assert PIN_REVIEW_EVIDENCE_REF, 'STOP: record where independent pin review was performed'
if MODE == 'EXECUTE_AUTHORIZED':
    assert REVIEWED_DATA_PINS is not None, 'STOP: independently reviewed pins required'
    assert PREVIOUS_PREPARATION_DIRECTORY, 'STOP: use the exact previously reviewed preparation'
    assert BIND_AUTHORIZATION_URL.startswith('https://github.com/') and APPROVE_TASK_URL.startswith('https://github.com/')
RUN_ID = datetime.now(ZoneInfo('Asia/Taipei')).strftime('%Y%m%d_%H%M%S_%f') + '_' + AGENT_SHA[:7]
LOCAL_SESSION = LOCAL_PARENT / RUN_ID
DRIVE_OUTPUT = DRIVE_OUTPUT_PARENT / RUN_ID
assert not LOCAL_SESSION.exists() and not DRIVE_OUTPUT.exists(), 'STOP: outputs must be new'
print('Mode:',MODE,'| snapshot:',SNAPSHOT_ID)
print('Existing archive:',RECOVERY_ARCHIVE)
print('New local session:',LOCAL_SESSION)
print('New exclusive research evidence archive:',DRIVE_OUTPUT)
print('No conversion / signing / automatic mode switch. Current cross-rate cost capability remains BLOCKED.')


In [ ]:
# Automated, pinned GitHub setup. No manual operations after the first cell.
import subprocess,sys,json,importlib,shutil
assert sys.version_info >= (3,11)
def clone_pin(repository,sha,directory):
    if not directory.exists():
        subprocess.run(['git','clone',repository,str(directory)],check=True)
        subprocess.run(['git','-C',str(directory),'checkout','--detach',sha],check=True)
    assert subprocess.check_output(['git','-C',str(directory),'rev-parse','HEAD'],text=True).strip()==sha
    assert not subprocess.check_output(['git','-C',str(directory),'status','--porcelain'],text=True).strip()
    return directory
AGENT_ROOT=clone_pin(AGENT_REPOSITORY,AGENT_SHA,Path('/content/strategy-research-agent-'+AGENT_SHA))
CORE_ROOT=clone_pin(CORE_REPOSITORY,CORE_SHA,Path('/content/market-research-core-'+CORE_SHA))
subprocess.run([sys.executable,'-m','pip','install','-e',str(CORE_ROOT),'-e',str(AGENT_ROOT)+'[research]'],check=True)
sys.path[:0]=[str(AGENT_ROOT/'src'),str(AGENT_ROOT),str(CORE_ROOT/'src')]
importlib.invalidate_caches()
from strategy_agent.execution import c1_integration as C
assert C.CORE==CORE_SHA and C.SNAPSHOT==SNAPSHOT_ID and C.AUTHORIZATION==NEW_AUTHORIZATION_ID
assert C.B.EXECUTOR==EXECUTOR_ID


In [ ]:
# Only the existing Development archive and its preserved approval evidence are read.
evidence=json.loads(ORIGINAL_APPROVAL_EVIDENCE_PATH.read_text())
approval=evidence['conversion_approval'] if APPROVAL_IS_IN_RECOVERY_RECORD else evidence
assert C.is_original_approval_url(ORIGINAL_CONVERSION_APPROVAL_REF)
assert C.is_original_approval_url(approval['approval_ref'])
inspection=C.inspect_archive(RECOVERY_ARCHIVE,conversion_approval=approval,trusted_pins=REVIEWED_DATA_PINS)
LOCAL_SESSION.mkdir(parents=True,exist_ok=False)
staged=C.stage_archive(inspection,LOCAL_SESSION/'staged')
print('Archive:',inspection['status'],'| local Core preflight:',staged['preflight']['status'])
print('Observed values (NOT automatically trusted):',json.dumps(inspection['observed_pins'],indent=2))
print('Producer:',inspection['original_producer_sha'])
print('Recovery implementation:',inspection['recovery_core_sha'],'| reader:',inspection['reader_core_sha'])


In [ ]:
# No automatic transition from preparation to execution; no signer/admin API is used.
OUTPUT=LOCAL_SESSION/'preparation'
blocked=None
try:
    if MODE=='PREPARE_ONLY':
        result=C.prepare(staged,OUTPUT,agent_commit=AGENT_SHA,root=AGENT_ROOT)
        print(json.dumps(result,indent=2))
    else:
        OUTPUT.mkdir(parents=True,exist_ok=False)
        # This currently refuses before budget because the pinned Core cannot switch dated costs.
        # After a separately reviewed Core delivery, the existing B3-C REAL_GATED chain
        # remains mandatory; no local switch may disable its authorization/approval/ledger gates.
        result=C.execute_authorized(Path(PREVIOUS_PREPARATION_DIRECTORY),
            governance_root=Path(GOVERNANCE_ROOT) if GOVERNANCE_ROOT else AGENT_ROOT,
            runtime_binding=staged['bundle'],bind_url=BIND_AUTHORIZATION_URL,task_url=APPROVE_TASK_URL,
            executor_id=EXECUTOR_ID)
except Exception as error:
    blocked=str(error)
    OUTPUT.mkdir(parents=True,exist_ok=True)
    C.save(OUTPUT/'BLOCKED.json',{'status':'BLOCKED','reason':blocked,'strategy_execution':'NOT_EXECUTED'})
finally:
    C.save(OUTPUT/'notebook_context.json',{'agent_commit':AGENT_SHA,'core_reader':CORE_SHA,'mode':MODE,
        'snapshot_id':SNAPSHOT_ID,'pin_review_evidence_ref':PIN_REVIEW_EVIDENCE_REF,
        'pin_summary':{k:v for k,v in inspection.items() if k not in ('bundle','snapshot_files')},
        'timezone':'Asia/Taipei','strategy_execution':'NOT_EXECUTED','consumed_batches':0,
        'authorization_created':False,'comments_posted':False})
    # Research evidence only. The local staged OHLCV file is not duplicated into this output.
    shutil.copytree(OUTPUT,DRIVE_OUTPUT)
    index={str(p.relative_to(DRIVE_OUTPUT)):C.sha(p) for p in sorted(DRIVE_OUTPUT.rglob('*')) if p.is_file()}
    for relative,expected in index.items():
        assert C.sha(OUTPUT/relative)==expected, 'STOP: Drive evidence copy differs from local'
    C.save(DRIVE_OUTPUT/'ARCHIVE_INDEX.json',{'files_sha256':index,'agent_commit':AGENT_SHA,'timezone':'Asia/Taipei'})
    print('New evidence archive:',DRIVE_OUTPUT)
if blocked:
    raise RuntimeError('STOPPED BEFORE RESEARCH: '+blocked)
print('STOP: preparation only. Personal review is required; current Core cost blocker prevents formal authorization readiness.')


## Stop — review, do not run conversion

Synthetic engineering tests are not real market execution. The user's earlier
Recovery PASS is data evidence; this notebook's actual archive hashes still need
independent pin review. Preserve the observed pin summary and review it before
entering REVIEWED_DATA_PINS in the first cell of a later run.

The current readiness is **not ready to submit formal authorization**: the pinned
Core cannot continuously apply the dated 2017–2024 SEC/TAF schedule. Resolve that
in a separately reviewed Core delivery and its PR dependencies first. Do not
broaden effective dates, use a 2024 fee throughout history or stitch monthly
backtests. Do not merge Core PRs from this notebook.

After all blockers are resolved, the lead must personally create the separate
one-batch, seven-day authorization and GitHub approvals through existing governance.
No APPROVE record, comment, batch, strategy backtest or Holdout access is generated
here. An archived local receipt remains nonportable; every fresh runtime must
stage locally and renew with Core again.
